# Database intro 

In [40]:
# importing the library
from peewee import *

In [41]:
# defining our database and giving it a name
db = SqliteDatabase('maryland_athletics.db')
# define the base model 
# this connects every model we define to the same database.
class BaseModel(Model):
    class Meta:
        database = db

## Let's create our first table!

In [42]:
class Team(BaseModel):
    name = CharField(unique=True)
    sport = CharField()
    season = CharField()
    class Meta:
        database = db

In [43]:
# connecting to our maryland_athletics.db database
db.connect()
# telling it to create the team table, which we've defined.
db.create_tables([Team])

In [44]:
# Let's take a look at this table. Is there anything in it?
for team in Team.select():
    print(team.name)

Maryland Women's Basketball
Maryland Men's Soccer
Maryland Football


In [45]:
# insert data into our table 
wbb_25, created = Team.get_or_create(
    name="Maryland Women's Basketball",
    defaults={
        "sport": "Women's Basketball",
        "season": "2025"
    }
)

msoc_25, created = Team.get_or_create(
    name="Maryland Men's Soccer",
    defaults={
        "sport": "Men's Soccer",
        "season": "2025"
    }
)

fb_25, created = Team.get_or_create(
    name="Maryland Football",
    defaults={
        "sport": "Football",
        "season": "2025"
    }
)

In [46]:
for team in Team.select():
    print(team.name)

Maryland Women's Basketball
Maryland Men's Soccer
Maryland Football


In [47]:
class Athlete(BaseModel):
    name = CharField()
    team = ForeignKeyField(Team, backref='athletes')
    athlete_year = CharField()
    athlete_position = CharField()
    class Meta:
        database = db

db.create_tables([Athlete])

In [48]:
malik_washington, created = Athlete.get_or_create(
    name="Malik Washington",
    defaults={
        "team": fb_25,
        "athlete_year": "Junior",
        "athlete_position": "Wide Receiver"
    }
)

In [49]:
for team in Team.select():
    print(team.name)

Maryland Women's Basketball
Maryland Men's Soccer
Maryland Football


In [50]:
class Game(BaseModel):
    date = DateField()
    opponent = CharField()
    home_away = CharField()     
    team = ForeignKeyField(Team, backref='games')
    class Meta:
        database = db
    

# stats are sport-agnostic — null fields just won't apply to every sport
class Stat(BaseModel):
    game = ForeignKeyField(Game, backref='stats')
    athlete = ForeignKeyField(Athlete, backref='stats')
    minutes_played = IntegerField(null=True)
    points = IntegerField(null=True)
    rebounds = IntegerField(null=True)
    assists = IntegerField(null=True)
    steals = IntegerField(null=True)
    turnovers = IntegerField(null=True)
    blocks = IntegerField(null=True)
    goals = IntegerField(null=True)
    shots = IntegerField(null=True)
    saves = IntegerField(null=True)
    class Meta:
        database = db

db.create_tables([Game, Stat])

In [51]:
# close the database connection when we're done
db.close()

True

## Now let's go over adding data using a spreadsheet and turning that into a database!


In [52]:
# read the spreadsheet
import pandas as pd
mcap_24 = pd.read_csv('data/data-9Tdi0.csv')

In [53]:
mcap_24

,District,School,Assessment,"Proficient, 2022","Proficient, 2023","Proficient, 2024",Percentage point change 2022-'23,Percentage point change 2023-'24
0,Montgomery,A. Mario Loiederman Middle,Mathematics 6,6,9.9,9.5,3.9,-0.4
1,Montgomery,A. Mario Loiederman Middle,Algebra 1,5,5,14,NaN,NaN
2,Montgomery,A. Mario Loiederman Middle,Geometry,5.7,9.3,14.3,3.6,5.0
3,Montgomery,A. Mario Loiederman Middle,English Language Arts 8,35.2,28.8,31.4,-6.4,2.6
4,Montgomery,A. Mario Loiederman Middle,English Language Arts 6,32.6,35.8,37.7,3.2,1.9
...,...,...,...,...,...,...,...,...
9086,Harford,Youths Benefit Elementary,Mathematics 4,47.8,52.4,56,4.6,3.6
9087,Harford,Youths Benefit Elementary,Mathematics 3,66.1,68,56,1.9,-12.0
9088,Harford,Youths Benefit Elementary,English Language Arts 5,58.8,59.2,67.6,0.4,8.4
9089,Harford,Youths Benefit Elementary,English Language Arts 3,66.9,70.7,67.9,3.8,-2.8


In [54]:
# get rid of the rows where district is called "seed" — these are just the average proficiency values for the state, not actual districts
mcap_24 = mcap_24[~mcap_24["District"].str.lower().eq("seed")]

## Clean up the data

In [55]:
# make the data long
long_df = mcap_24.melt(
    id_vars=["District", "School", "Assessment"],
    value_vars=[
        "Proficient, 2022",
        "Proficient, 2023",
        "Proficient, 2024"
    ],
    var_name="year",
    value_name="proficient_percent"
)

In [56]:
long_df

,District,School,Assessment,year,proficient_percent
0,Montgomery,A. Mario Loiederman Middle,Mathematics 6,"Proficient, 2022",6
1,Montgomery,A. Mario Loiederman Middle,Algebra 1,"Proficient, 2022",5
2,Montgomery,A. Mario Loiederman Middle,Geometry,"Proficient, 2022",5.7
3,Montgomery,A. Mario Loiederman Middle,English Language Arts 8,"Proficient, 2022",35.2
4,Montgomery,A. Mario Loiederman Middle,English Language Arts 6,"Proficient, 2022",32.6
...,...,...,...,...,...
27238,Harford,Youths Benefit Elementary,Mathematics 4,"Proficient, 2024",56
27239,Harford,Youths Benefit Elementary,Mathematics 3,"Proficient, 2024",56
27240,Harford,Youths Benefit Elementary,English Language Arts 5,"Proficient, 2024",67.6
27241,Harford,Youths Benefit Elementary,English Language Arts 3,"Proficient, 2024",67.9


In [57]:
# create a new year column based on the old year column
long_df["year"] = long_df["year"].astype(str).str.extract(r'(\d{4})').astype(int)

In [58]:
long_df["proficient_percent"] = (
    long_df["proficient_percent"]
    .replace("*", None)
    .astype(float)
)

In [59]:
long_df

,District,School,Assessment,year,proficient_percent
0,Montgomery,A. Mario Loiederman Middle,Mathematics 6,2022,6.0
1,Montgomery,A. Mario Loiederman Middle,Algebra 1,2022,5.0
2,Montgomery,A. Mario Loiederman Middle,Geometry,2022,5.7
3,Montgomery,A. Mario Loiederman Middle,English Language Arts 8,2022,35.2
4,Montgomery,A. Mario Loiederman Middle,English Language Arts 6,2022,32.6
...,...,...,...,...,...
27238,Harford,Youths Benefit Elementary,Mathematics 4,2024,56.0
27239,Harford,Youths Benefit Elementary,Mathematics 3,2024,56.0
27240,Harford,Youths Benefit Elementary,English Language Arts 5,2024,67.6
27241,Harford,Youths Benefit Elementary,English Language Arts 3,2024,67.9


In [60]:
# drop rows with missing proficiency values
long_df = long_df.dropna(subset=["proficient_percent"])

In [61]:
# create a new year column based on the old year column
long_df["year"] = long_df["year"].astype(str).str.extract(r'(\d{4})').astype(int)

In [62]:
# create a new database for the mcap data
db = SqliteDatabase("mcap.db")

# define the schemas for all of our tables!
class BaseModel(Model):
    class Meta:
        database = db
# district table
class District(BaseModel):
    name = CharField(unique=True)
# school table
class School(BaseModel):
    name = CharField()
    district = ForeignKeyField(District, backref="schools")

    class Meta:
        indexes = (
            (("name", "district"), True),
        )
# assessment table
class Assessment(BaseModel):
    name = CharField(unique=True)
# result table
class Result(BaseModel):
    school = ForeignKeyField(School, backref="results")
    assessment = ForeignKeyField(Assessment, backref="results")
    year = IntegerField()
    proficient_percent = FloatField()

    class Meta:
        indexes = (
            (("school", "assessment", "year"), True),
        )

In [63]:
# create the tables
db.connect()
db.create_tables([District, School, Assessment, Result])

In [64]:
# insert districts
for district_name in long_df["District"].unique():
    District.get_or_create(name=district_name)
# insert schools
for _, row in long_df[["School", "District"]].drop_duplicates().iterrows():
    district = District.get(District.name == row["District"])

    School.get_or_create(
        name=row["School"],
        district=district
    )
# insert assessments
for assessment_name in long_df["Assessment"].unique():
    Assessment.get_or_create(name=assessment_name)
# insert results
for _, row in long_df.iterrows():
    school = School.get(School.name == row["School"])
    assessment = Assessment.get(Assessment.name == row["Assessment"])

    Result.get_or_create(
        school=school,
        assessment=assessment,
        year=row["year"],
        defaults={"proficient_percent": row["proficient_percent"]}
    )

KeyboardInterrupt: 

# Your turn!!!
For your homework, create a relational database for any data of your choice. There must be at least three tables and at least 10 rows in each table. You may either manually add the data or turn a spreadsheet into a relational database. Use the peewee package we went over in class. LLMS/AI are your friend! Use GitHub Copilot if you get stuck.

In [ ]:
# read the spreadsheet
import pandas as pd
measles = pd.read_csv('data/4zhkG.csv')

In [ ]:
measles

,State,Imported,Local,Total,Recent 2 Weeks,2025,2026
0,U.S.,211,3307,3517,190,2213,1304
1,South Carolina,5,989,994,18,302,692
2,Texas,1,829,830,27,803,27
3,Utah,0,358,358,58,156,202
4,Arizona,4,272,276,7,220,56
5,Florida,11,121,131,32,7,124
6,New Mexico,1,105,106,6,100,6
7,Kansas,0,91,91,0,91,0
8,North Dakota,6,53,59,6,36,23
9,Ohio,7,47,54,0,45,9


In [66]:
# create a new database for the measles data
db = SqliteDatabase("measles.db")

# define the schemas for all of our tables!
class BaseModel(Model):
    class Meta:
        database = db

# state table
class State(BaseModel):
    name = CharField(unique=True)

# case type table (imported, local, total)
class CaseType(BaseModel):
    name = CharField(unique=True)

# measles case table
class MeaslesCase(BaseModel):
    state = ForeignKeyField(State, backref="cases")
    case_type = ForeignKeyField(CaseType, backref="cases")
    time_period = CharField()  # "Recent 2 Weeks", "2025", "2026"
    count = IntegerField()

    class Meta:
        indexes = (
            (("state", "case_type", "time_period"), True),
        )

In [68]:
# create the tables
db.connect()
db.create_tables([State, CaseType, MeaslesCase])

OperationalError: Connection already opened.

In [69]:
# First, handle the case type columns (Imported, Local, Total)
case_type_cols = ["Imported", "Local", "Total"]
measles_cases = measles.melt(
    id_vars=["State"],
    value_vars=case_type_cols,
    var_name="case_type",
    value_name="count"
)
measles_cases["time_period"] = "All Years"  # These are totals across all time

# Then handle the time period columns
time_period_cols = ["Recent 2 Weeks", "2025", "2026"]
measles_periods = measles.melt(
    id_vars=["State"],
    value_vars=time_period_cols,
    var_name="time_period",
    value_name="count"
)
measles_periods["case_type"] = "Total"  # These are total cases for the time period

# Combine both datasets
measles_long = pd.concat([measles_cases, measles_periods], ignore_index=True)

# insert states
for state_name in measles_long["State"].unique():
    State.get_or_create(name=state_name)

# insert case types
for case_type_name in measles_long["case_type"].unique():
    CaseType.get_or_create(name=case_type_name)

# insert measles cases
for _, row in measles_long.iterrows():
    state = State.get(State.name == row["State"])
    case_type = CaseType.get(CaseType.name == row["case_type"])

    MeaslesCase.get_or_create(
        state=state,
        case_type=case_type,
        time_period=row["time_period"],
        defaults={"count": int(row["count"])}
    )

Yay! It's done!